# Fase 8 — Evaluación del producto

Mide las tres piezas nuevas del pivot con el `eval/casos_prueba.csv` anotado a mano por el
equipo (17 casos coloquiales con las leyes esperadas):

1. **Precision@K / Recall del retrieval** del chatbot — con y sin *query rewriting*, para
   justificar esa decisión de diseño con un número.
2. **Trazabilidad**: % de citas del chatbot que pasan la validación automática de quotes.
3. **Calidad de resúmenes de vínculos**: muestra estratificada para revisión humana (mismo
   enfoque que el golden set de la Fase 3).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "src" / "lexar").exists():
        sys.path.insert(0, str(_base / "src"))
        break

import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 220)

from lexar import config
from lexar.chatbot import answer_case, rewrite_query
from lexar.embeddings import embed_texts
from lexar.links import load_norm_links
from lexar.rate_limiter import AdaptiveRateLimiter
from lexar.retrieval import aggregate_hits_by_document, load_case_index, load_law_index

TOP_K = 10  # Precision@K / Recall@K sobre documentos de ley

assert config.CASOS_PRUEBA_PATH.exists(), f"Falta {config.CASOS_PRUEBA_PATH}"
casos = pd.read_csv(config.CASOS_PRUEBA_PATH)
casos["leyes_esperadas"] = casos["leyes_esperadas"].str.split(";")
print(f"Casos de prueba: {len(casos)}")
casos.head(3)

## 8.1 Precision@K / Recall — con y sin query rewriting

Ablation barata: para cada caso, se compara el retrieval usando la consulta cruda (el texto
coloquial tal cual) contra usar las consultas reescritas por el LLM en lenguaje jurídico. Se
matchea contra `leyes_esperadas` por número de norma (substring en `titulo_resumido`/`document_id`
no alcanza; se usa el mapeo `numero_norma → document_id` de `document_identifiers.csv`).

In [ ]:
identifiers = pd.read_csv(
    config.DATASET_DIR / "infoleg" / "procesado" / "document_identifiers.csv",
    dtype=str, keep_default_na=False,
)
numero_to_docs = identifiers.groupby("numero_norma")["document_id"].apply(set).to_dict()

law_index = load_law_index()
limiter = AdaptiveRateLimiter()


def retrieved_document_ids(query_texts: list[str], top_k: int = TOP_K) -> list[str]:
    vectors = embed_texts(query_texts, limiter)
    hits = law_index.search(vectors, top_k=top_k * 3)
    docs = aggregate_hits_by_document(hits, "document_id").head(top_k)
    return docs["document_id"].tolist()


def precision_recall(retrieved: list[str], expected_numeros: list[str]) -> tuple[float, float]:
    expected_docs = set().union(*(numero_to_docs.get(n.strip(), set()) for n in expected_numeros))
    if not expected_docs:
        return float("nan"), float("nan")
    hit = len(set(retrieved) & expected_docs)
    precision = hit / len(retrieved) if retrieved else 0.0
    recall = hit / len(expected_docs)
    return precision, recall

In [ ]:
rows = []
for _, caso in casos.iterrows():
    raw_retrieved = retrieved_document_ids([caso["caso"]])
    rewrite = rewrite_query(caso["caso"], limiter)
    rewritten_retrieved = retrieved_document_ids(rewrite["consultas"])

    p_raw, r_raw = precision_recall(raw_retrieved, caso["leyes_esperadas"])
    p_rw, r_rw = precision_recall(rewritten_retrieved, caso["leyes_esperadas"])
    rows.append({
        "caso": caso["caso"][:60], "precision_sin_rewrite": p_raw, "recall_sin_rewrite": r_raw,
        "precision_con_rewrite": p_rw, "recall_con_rewrite": r_rw,
    })

ablation = pd.DataFrame(rows)
print(f"Precision@{TOP_K} promedio — sin rewrite: {ablation['precision_sin_rewrite'].mean():.1%} | con rewrite: {ablation['precision_con_rewrite'].mean():.1%}")
print(f"Recall@{TOP_K} promedio — sin rewrite: {ablation['recall_sin_rewrite'].mean():.1%} | con rewrite: {ablation['recall_con_rewrite'].mean():.1%}")
ablation

## 8.2 Trazabilidad de citas del chatbot

Corre `answer_case()` completo (con fallos CSJN si la Fase 4 ya está disponible) sobre todos los
casos y mide qué fracción de las citas devueltas son verificables textualmente contra la fuente
citada — la misma validación que usa la app en vivo.

In [ ]:
case_index = load_case_index() if config.CASE_EMBEDDINGS_NPY_PATH.exists() else None
if case_index is None:
    print("Aviso: sin índice de fallos (correr la Fase 4 primero); se evalúa solo con leyes.")

citation_rows = []
for _, caso in casos.iterrows():
    result = answer_case(caso["caso"], law_index, case_index, limiter=limiter)
    for citation in result["citas"]:
        citation_rows.append({"caso": caso["caso"][:60], **citation})

citations_df = pd.DataFrame(citation_rows)
trazabilidad = citations_df["cita_valida"].mean() if len(citations_df) else float("nan")
print(f"Citas totales: {len(citations_df)} | trazabilidad (citas válidas): {trazabilidad:.1%}")
citations_df[~citations_df["cita_valida"]].head(10) if len(citations_df) else citations_df

## 8.3 Muestra para rúbrica humana de resúmenes de vínculos

Estratificada por `link_source` × `dominant_label`, igual que el golden set de la Fase 3.
Completá `human_rubrica` a mano (correcto / útil / citas_verificables) y volvé a correr la
celda siguiente para el resumen agregado.

In [ ]:
from lexar.summaries import LinkSummarizer

RUBRICA_SAMPLE_SIZE = 30
RUBRICA_PATH = config.EVAL_DIR / "rubrica_resumenes.csv"
config.EVAL_DIR.mkdir(parents=True, exist_ok=True)

norm_links = load_norm_links()
strata = norm_links.dropna(subset=["dominant_label"])
sample_parts = [
    group.sample(n=min(len(group), 3), random_state=config.RANDOM_SEED)
    for _, group in strata.groupby(["link_source", "dominant_label"], observed=True)
]
sample = pd.concat(sample_parts).head(RUBRICA_SAMPLE_SIZE).copy() if sample_parts else strata.iloc[0:0]

summarizer = LinkSummarizer()
sample["relacion"] = ""
sample["relevancia_juridica"] = ""
for idx, row in sample.iterrows():
    summary = summarizer.summarize(row["document_id_a"], row["document_id_b"], row)
    sample.loc[idx, "relacion"] = summary["relacion"]
    sample.loc[idx, "relevancia_juridica"] = summary["relevancia_juridica"]

sample["human_rubrica"] = ""  # completar a mano: correcto / parcial / incorrecto
sample[[
    "doc_pair_key", "link_source", "dominant_label", "relacion", "relevancia_juridica", "human_rubrica",
]].to_csv(RUBRICA_PATH, index=False, encoding="utf-8")
print(f"Guardado: {RUBRICA_PATH} ({len(sample)} vínculos para revisar a mano)")

In [ ]:
if RUBRICA_PATH.exists():
    labeled = pd.read_csv(RUBRICA_PATH)
    labeled = labeled[labeled["human_rubrica"].notna() & (labeled["human_rubrica"].astype(str).str.strip() != "")]
    if labeled.empty:
        print("Todavía no se completó 'human_rubrica' en el CSV. Etiquetar y volver a correr esta celda.")
    else:
        print(labeled["human_rubrica"].value_counts(normalize=True).round(3))
else:
    print("Correr la celda anterior primero.")